# Natural Language Processing Notebook

## Topic: Context-Free Grammars in NLP

## 1. What is a Context-Free Grammar?

### Definition

A **Context-Free Grammar (CFG)** is a formal system for describing the syntax of a
language. It consists of four parts:

* A set of **terminals** — the actual words (`the`, `dog`, `barked`)
* A set of **non-terminals** — syntactic categories (`S`, `NP`, `VP`, `Det`, `N`, `V`)
* A **start symbol** — the non-terminal every valid sentence must reduce to (usually `S`)
* A set of **production rules** — how non-terminals expand, e.g. `NP -> Det N`

It is called "context-free" because every rule rewrites a single non-terminal
regardless of what surrounds it — the rule `NP -> Det N` applies the same way whether
the `NP` sits at the start of the sentence or three clauses deep.

This is the formal grammar that the `parsing.ipynb` notebook's recursive-descent and
shift-reduce parsers were built to consume. Here we use `nltk`'s CFG tools instead of
hand-rolling the grammar dictionary, and look at what a CFG can and cannot express.

In [ ]:
import nltk
from nltk import CFG
from nltk.parse.generate import generate

print("nltk version:", nltk.__version__)

## 2. Defining a grammar

`nltk.CFG.fromstring` takes a block of rules written in a simple text notation:
`LHS -> RHS1 | RHS2 | ...`. Terminals are quoted; non-terminals are bare identifiers.

In [ ]:
grammar = CFG.fromstring('''
    S -> NP VP
    NP -> Det N | Det N PP | 'John' | 'Mary'
    VP -> V NP | V NP PP | V
    PP -> P NP
    Det -> 'the' | 'a' | 'my'
    N -> 'dog' | 'cat' | 'telescope' | 'park' | 'man'
    V -> 'saw' | 'walked' | 'barked'
    P -> 'in' | 'with'
''')

print(grammar)

In [ ]:
print("Start symbol:", grammar.start())
print("\nProductions:")
for prod in grammar.productions():
    print(" ", prod)

## 3. Parsing sentences against the grammar

`nltk.ChartParser` builds every valid parse tree for a tokenised sentence using dynamic
programming (a chart), which is far more efficient than the plain recursive-descent
parser from `parsing.ipynb` — it never re-derives the same sub-tree twice.

In [ ]:
parser = nltk.ChartParser(grammar)

sentence = "the dog saw a cat".split()
trees = list(parser.parse(sentence))

print(f"'{' '.join(sentence)}' -> {len(trees)} parse(s) found\n")
for t in trees:
    print(t)

In [ ]:
# Pretty-print the tree structure
if trees:
    trees[0].pretty_print()

## 4. A sentence the grammar rejects

A CFG defines a language *exactly* — anything not derivable from the rules is rejected.
This is a feature, not a limitation: it is how a parser can tell you a sentence is
ungrammatical rather than silently guessing.

In [ ]:
bad_sentence = "dog the saw cat a".split()          # words scrambled out of order
trees_bad = list(parser.parse(bad_sentence))
print(f"'{' '.join(bad_sentence)}' -> {len(trees_bad)} parse(s) found")

not_in_vocab = "the dog saw a telescope with my cat".split()
trees_ok = list(parser.parse(not_in_vocab))
print(f"'{' '.join(not_in_vocab)}' -> {len(trees_ok)} parse(s) found")
for t in trees_ok:
    print(t)

## 5. Ambiguity: more than one valid parse

The sentence above (`"the dog saw a telescope with my cat"`) is a classic example of
**prepositional-phrase attachment ambiguity**: the `PP -> with my cat` can attach either
to the `NP` (`a telescope with my cat` — a telescope that has a cat with it, i.e. modifies
"telescope") or to the `VP` (`saw ... with my cat` — the seeing was done alongside the
cat). Both readings are grammatically valid under this CFG, so the parser returns both
trees. Humans resolve this kind of ambiguity with world knowledge and context — a CFG
alone cannot.

In [ ]:
print(f"Number of parses: {len(trees_ok)}\n")
for i, t in enumerate(trees_ok, 1):
    print(f"--- parse {i} ---")
    t.pretty_print()

## 6. Generating sentences from a grammar

A CFG does not just recognise sentences — it can *produce* every sentence in its
language. `nltk.parse.generate.generate` enumerates them (careful with recursive
grammars: the language can be infinite, so always pass a `depth` or `n` limit).

In [ ]:
print("All sentences this grammar can generate (capped at the first 20):\n")
for i, sent in enumerate(generate(grammar, n=20), 1):
    print(f"{i:>3}. {' '.join(sent)}")

## 7. Recursion and infinite languages

Real CFGs for natural language are recursive: a `NP` can contain a `PP`, which contains
another `NP`, which can contain another `PP`, without limit
(*"the cat in the hat in the box in the ... "*). This is exactly why the `depth`
parameter matters — without it, `generate` would never terminate on a recursive grammar.

In [ ]:
recursive_grammar = CFG.fromstring('''
    NP -> Det N | Det N PP
    PP -> P NP
    Det -> 'the'
    N -> 'cat' | 'hat' | 'box'
    P -> 'in'
''')

print("Sentences up to depth 4 (i.e. at most 1 level of PP recursion):")
for sent in generate(recursive_grammar, depth=4):
    print(" ", " ".join(sent))

print("\nSentences up to depth 6 (recursion goes one level deeper):")
for sent in generate(recursive_grammar, depth=6):
    print(" ", " ".join(sent))

print("\nWithout a depth limit, generate() would try to enumerate an infinite language --")
print("there is no longest noun phrase in English, only longer and longer ones.")

## 8. From CFG to the hand-written parsers in `parsing.ipynb`

The `parsing.ipynb` notebook implements a **Recursive Descent Parser** and a
**Shift-Reduce Parser** directly against a grammar dictionary. Both are just two
different *algorithms* for answering the same question a CFG poses: *"can this sequence
of terminals be derived from the start symbol using these production rules?"*

| | Recursive Descent | Shift-Reduce | `nltk.ChartParser` |
|---|---|---|---|
| Direction | Top-down | Bottom-up | Bottom-up, tabulated |
| Backtracking | Yes, can be exponential | Yes, can get stuck ("shift-reduce conflict") | No — every sub-parse is cached |
| Finds all parses? | With modification | With modification | Yes, by default |
| Practical for real grammars? | Small grammars only | Small grammars only | Scales far better |

This is the same trade-off you will see again in dynamic programming problems outside
NLP: naive recursion re-does work, and a chart (memo table) avoids it.

## Exercises

**Exercise 1.** Extend the first grammar so it can also parse `"John gave the man a
telescope"` (a ditransitive verb taking two objects). Verify with the parser.

In [ ]:
# --- Solution 1 ---------------------------------------------------------------
grammar_ex1 = CFG.fromstring('''
    S -> NP VP
    NP -> Det N | Det N PP | 'John' | 'Mary'
    VP -> V NP | V NP PP | V | V NP NP
    PP -> P NP
    Det -> 'the' | 'a' | 'my'
    N -> 'dog' | 'cat' | 'telescope' | 'park' | 'man'
    V -> 'saw' | 'walked' | 'barked' | 'gave'
    P -> 'in' | 'with'
''')

parser_ex1 = nltk.ChartParser(grammar_ex1)
sent = "John gave the man a telescope".split()
trees_ex1 = list(parser_ex1.parse(sent))
print(f"'{' '.join(sent)}' -> {len(trees_ex1)} parse(s)")
for t in trees_ex1:
    t.pretty_print()

**Exercise 2.** Write a small function `is_grammatical(grammar, sentence)` that returns
`True`/`False` for whether a sentence has at least one valid parse, and use it to check
five sentences at once.

In [ ]:
# --- Solution 2 ---------------------------------------------------------------
def is_grammatical(grammar, sentence):
    parser = nltk.ChartParser(grammar)
    tokens = sentence.split()
    try:
        return len(list(parser.parse(tokens))) > 0
    except ValueError:
        # a word not covered by any terminal in the grammar
        return False

test_sentences = [
    "the dog saw a cat",
    "a cat barked",
    "dog the saw",
    "John walked",
    "the dog saw a robot",     # 'robot' is not in the vocabulary
]

for s in test_sentences:
    print(f"{'YES' if is_grammatical(grammar, s) else 'no ':<4} -- {s}")

**Exercise 3.** Count how many distinct sentences of length exactly 3 words the
`recursive_grammar` from section 7 can generate, without hardcoding the answer.

In [ ]:
# --- Solution 3 ---------------------------------------------------------------
three_word = [s for s in generate(recursive_grammar, depth=10) if len(s) == 3]
print(f"{len(three_word)} distinct 3-word sentences:")
for s in three_word:
    print(" ", " ".join(s))

**Exercise 4 (challenge).** The grammar in this notebook treats `"saw"` only as a verb
(as in "the dog saw a cat"). Add a rule so the SAME grammar can also parse `"the saw"` as
a noun phrase (a hand tool), then explain, using the parser's own output, how the
grammar now behaves on the ambiguous single word `"saw"`.

In [ ]:
# --- Solution 4 ---------------------------------------------------------------
grammar_ex4 = CFG.fromstring('''
    S -> NP VP
    NP -> Det N | Det N PP | 'John' | 'Mary'
    VP -> V NP | V NP PP | V
    PP -> P NP
    Det -> 'the' | 'a' | 'my'
    N -> 'dog' | 'cat' | 'telescope' | 'park' | 'man' | 'saw'
    V -> 'saw' | 'walked' | 'barked'
    P -> 'in' | 'with'
''')

parser_ex4 = nltk.ChartParser(grammar_ex4)

sent_np = "the saw".split()
trees_np = list(parser_ex4.parse(sent_np))
print(f"'{' '.join(sent_np)}' -> {len(trees_np)} parse(s)")
for t in trees_np:
    print(t)

sent_full = "the dog saw the saw".split()
trees_full = list(parser_ex4.parse(sent_full))
print(f"\n'{' '.join(sent_full)}' -> {len(trees_full)} parse(s)")
for t in trees_full:
    t.pretty_print()

print("Explanation: 'saw' is now BOTH a verb (V) and a noun (N) in the grammar. The word")
print("by itself is genuinely ambiguous in English too -- 'the saw' only resolves to the")
print("tool because 'the' can only precede a noun, not a verb. The grammar captures that")
print("constraint automatically through the production rules, exactly the way a human")
print("reader resolves it from local syntactic context.")

## Summary

| Concept | Key point |
|---|---|
| CFG | Terminals, non-terminals, a start symbol, and production rules |
| "Context-free" | Every rule rewrites one non-terminal regardless of surrounding context |
| Parsing | Given a grammar and a sentence, find derivation(s) from the start symbol |
| `ChartParser` | Bottom-up, memoised — avoids the repeated work of naive recursive descent |
| Ambiguity | More than one valid parse tree for the same sentence — a CFG cannot resolve it alone |
| Generation | A grammar defines a language; `generate()` enumerates it (bound the depth!) |
| Recursion | Recursive rules (`NP -> Det N PP`, `PP -> P NP`) make the language infinite |

See `parsing.ipynb` in this same folder for hand-implemented recursive-descent and
shift-reduce parsers built directly against a grammar dictionary, without `nltk`.